# Data Prep for Time to Injury Forecasting DeepHit Model

This script replicates this paper https://arxiv.org/html/2601.19479v1 that builds a time-to-injury forecasting model based on the DeebHit model and uses SoccerMon data

In [8]:
from subjective_metrics_pipeline import PlayerMetricsPipeline
from metrics_eda import MetricsEDA
from GPS_feature_extractor import GPSFeatureExtractor

import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno 
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, StandardScaler

## 1. Subjective Data Prep

In [30]:
# 1. Your exact, verified file paths
file_mapping = {
    'data/subjective/training-load/atl.csv': 'ATL',
    'data/subjective/training-load/weekly_load.csv': 'Weekly load',
    'data/subjective/training-load/monotony.csv': 'Monotony',
    'data/subjective/training-load/strain.csv': 'Strain',
    'data/subjective/training-load/acwr.csv': 'ACWR',
    'data/subjective/training-load/ctl28.csv': 'CTL28',
    'data/subjective/training-load/ctl42.csv': 'CTL42',
    'data/subjective/wellness/fatigue.csv': 'Fatigue',
    'data/subjective/wellness/mood.csv': 'Mood',
    'data/subjective/wellness/readiness.csv': 'Readiness',
    'data/subjective/wellness/sleep_duration.csv': 'Sleep duration',
    'data/subjective/wellness/soreness.csv': 'Soreness',
    'data/subjective/wellness/stress.csv': 'Stress'
}

# 3. Instantiate WITH the mapping dictionary explicitly
pipeline = PlayerMetricsPipeline(mapping=file_mapping)
final_df = pipeline.run()

# Display the first few rows
final_df.head()

🚀 Starting pipeline execution...

  ✅ Processed ATL: Generated 36550 rows.
  ✅ Processed Weekly load: Generated 36550 rows.
  ✅ Processed Monotony: Generated 36550 rows.
  ✅ Processed Strain: Generated 36550 rows.
  ✅ Processed ACWR: Generated 36550 rows.
  ✅ Processed CTL28: Generated 36550 rows.
  ✅ Processed CTL42: Generated 36550 rows.
  ✅ Processed Fatigue: Generated 36550 rows.
  ✅ Processed Mood: Generated 36550 rows.
  ✅ Processed Readiness: Generated 36550 rows.
  ✅ Processed Sleep duration: Generated 36550 rows.
  ✅ Processed Soreness: Generated 36550 rows.
  ✅ Processed Stress: Generated 36550 rows.

⏳ Formatting final schema. Total rows merged: 36550...
🎉 Pipeline complete! Final dataset shape: (36550, 15)


,player_name,date,ATL,Weekly load,Monotony,Strain,ACWR,CTL28,CTL42,Fatigue,Mood,Readiness,Sleep duration,Soreness,Stress
0,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,2020-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,2020-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,2020-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,2020-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,2020-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Save to the current folder as a Parquet file
final_df.to_parquet("data/master_subjective_features.parquet", index=False)

print("✅ Saved as Parquet successfully!")

### 1.1 EDA (Apply after all joins)

In [ ]:
# # using team B only
# team_b_df = final_df[final_df["player_name"].str.contains("TeamB", na=False)]

# # Assuming 'final_df' is the dataframe outputted from your pipeline in the previous step:

# eda_pipeline = MetricsEDA(df=team_b_df)
# multivariate_anomalies = eda_pipeline.run_all()

# # You can inspect the specific abnormal dates/players identified by the Isolation Forest:
# display(multivariate_anomalies.head())
# display(multivariate_anomalies.shape)

In [ ]:
# def diagnose_missingness(df):
#     print("Running Missingness Diagnostics...\n")
    
#     subjective_cols = ['Fatigue', 'Mood', 'Readiness', 'Sleep duration', 'Soreness', 'Stress']
    
#     # 1. Player Compliance Check (Bar Chart)
#     compliance = df.groupby('player_name')[subjective_cols].apply(lambda x: x.notnull().mean().mean() * 100).sort_values()
    
#     plt.figure(figsize=(12, 6))
#     sns.barplot(x=compliance.values, y=compliance.index, palette="flare")
#     plt.title("Player Compliance: Average % of Wellness Forms Completed", fontsize=14)
#     plt.xlabel("Completion Percentage (%)")
#     plt.ylabel("Player ID")
#     plt.tight_layout()
#     plt.show()

#     # 2. Temporal Missingness Heatmap     
#     # Ensure date is a datetime object first
#     df_sorted = df.copy()
#     df_sorted['date'] = pd.to_datetime(df_sorted['date'])
    
#     # Sort chronologically, set as index, and convert to PeriodIndex
#     df_time_indexed = df_sorted.sort_values('date').set_index('date')
#     df_time_indexed.index = df_time_indexed.index.to_period('D')
    
#     plt.figure(figsize=(12, 6))
#     msno.matrix(df_time_indexed[subjective_cols], sparkline=False)
#     plt.title("Missing Data Over Time (White = Missing, Dark = Present)", fontsize=16)
#     plt.show()

# # Run it on your dataframe
# diagnose_missingness(final_df)

### 1.2 Feature Engineering + dealing with EDA outcomes

In [ ]:
# def clip_raw_outliers(df: pd.DataFrame, columns: list, lower_q: float = 0.01, upper_q: float = 0.99) -> pd.DataFrame:
#     """
#     Winsorizes raw numerical features at specified quantiles to prevent 
#     extreme outliers from corrupting downstream standardization math.
#     """
#     df_clipped = df.copy()
#     print(f"Applying percentile clipping (Lower: {lower_q}, Upper: {upper_q})...")
    
#     for col in columns:
#         if col not in df_clipped.columns:
#             continue
            
#         # Calculate thresholds, ignoring NaNs
#         lower_bound = df_clipped[col].quantile(lower_q)
#         upper_bound = df_clipped[col].quantile(upper_q)
        
#         # Diagnostics
#         clipped_low = (df_clipped[col] < lower_bound).sum()
#         clipped_high = (df_clipped[col] > upper_bound).sum()
        
#         if clipped_low > 0 or clipped_high > 0:
#             print(f"  -> {col}: Capped {clipped_low} low | {clipped_high} high values.")
            
#         # Apply the clip
#         df_clipped[col] = df_clipped[col].clip(lower=lower_bound, upper=upper_bound)
        
#     return df_clipped


# def engineer_subjective_features_for_mlp(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Transforms and scales features specifically for a Neural Network backbone,
#     handles zero-inflation, creates missingness masks, clips outliers, and safely imputes for tensors.
#     """
#     print("Pre-processing data for MLP backbone...\n" + "-"*40)
#     df_dl = df.copy()
    
#     # 1. Define Feature Groups
#     skewed_load_cols = ['ATL', 'Weekly load', 'Monotony', 'Strain', 'CTL28', 'CTL42']
#     standard_cols = ['ACWR', 'Fatigue', 'Mood', 'Readiness', 'Sleep duration', 'Soreness', 'Stress']
#     wellness_cols = ['Fatigue', 'Mood', 'Readiness', 'Sleep duration', 'Soreness', 'Stress']
    
#     # Ensure columns exist before operating on them
#     skewed_load_cols = [col for col in skewed_load_cols if col in df_dl.columns]
#     standard_cols = [col for col in standard_cols if col in df_dl.columns]
#     wellness_cols = [col for col in wellness_cols if col in df_dl.columns]

#     # 2. Engineer Missingness Mask (The Deep Learning Signal)
#     if wellness_cols:
#         df_dl['wellness_reported'] = df_dl[wellness_cols].notnull().any(axis=1).astype(int)
#         print("✅ Created 'wellness_reported' mask feature.")

#     # 3. Extract "Rest Day" Signal before transforming zeros away
#     if 'Daily load' in df_dl.columns:
#         df_dl['is_rest_day'] = (df_dl['Daily load'] == 0).astype(int)
#     elif 'Strain' in df_dl.columns:
#         df_dl['is_rest_day'] = (df_dl['Strain'] == 0).astype(int)
        
#     if 'is_rest_day' in df_dl.columns:
#         print("✅ Created 'is_rest_day' feature.")

#     # ---------------------------------------------------------
#     # NEW STEP: 4. Clip Raw Outliers BEFORE Standardizing
#     # We clip standard columns here. Skewed load cols don't need it 
#     # because Yeo-Johnson inherently handles their extreme tails.
#     # ---------------------------------------------------------
#     if standard_cols:
#         df_dl = clip_raw_outliers(df_dl, columns=standard_cols, lower_q=0.01, upper_q=0.99)
#         print("✅ Winsorized raw standard features to protect scaling math.")

#     # 5. Transform Skewed Load Features (Yeo-Johnson)
#     if skewed_load_cols:
#         power_scaler = PowerTransformer(method='yeo-johnson', standardize=True)
#         df_dl[skewed_load_cols] = power_scaler.fit_transform(df_dl[skewed_load_cols])
#         print(f"✅ Applied Yeo-Johnson transform & scaling to {len(skewed_load_cols)} load metrics.")

#     # 6. Standardize Remaining Features (Z-Score)
#     if standard_cols:
#         standard_scaler = StandardScaler()
#         # StandardScaler automatically ignores NaNs during fit/transform
#         df_dl[standard_cols] = standard_scaler.fit_transform(df_dl[standard_cols])
#         print(f"✅ Applied Standard Scaling to {len(standard_cols)} standard metrics.")

#     # 7. Re-apply the Deep Learning Mask for Missing Values
#     # Since StandardScaler shifted the mean to 0, filling NaNs with 0 is perfect mean-imputation.
#     if standard_cols:
#         df_dl[standard_cols] = df_dl[standard_cols].fillna(0)
    
#     # Ensure load columns don't have stray NaNs breaking the tensors
#     if skewed_load_cols:
#         df_dl[skewed_load_cols] = df_dl[skewed_load_cols].fillna(0)
        
#     print("✅ Applied safe null masking (0-fill) for MLP tensors.\n" + "-"*40)

#     return df_dl

# # --- Execution ---
# # Now you just pass your raw, un-imputed daily dataframe straight into this function:
# deephit_ready_df = engineer_subjective_features_for_mlp(final_df)
# deephit_ready_df.head()

In [ ]:
# # Checkin EDA plots post data processing
# eda_pipeline = SubjectiveMetricsEDA(df=deephit_ready_df)
# multivariate_anomalies = eda_pipeline.run_all()

Steps taken for subjective data:

- Loaded in wide data and transfored to long version based on player_name and date
- Performed EDA to see proportions of missing data (missingness), feature distributions, box plots for outliers, correlation matrix and longitudinal team trends.
- To deal with missingess we used missingno pattern to find if the missingness is temporal (certain periods where all players are responsive) or individual (some players partake in the study while others don't).
- An engineered feature was created wellness_reported (0 or 1) to account for lazy individuals.
- Fixing feature distributions was to use Yeo-Johnson transformation for skewed data and standardscaler transform for standard columns.
- Outliers were dealt with by using winsorized clipping between the 1 and 55 percentiles


## 2. GPS Based Data Prep

In [ ]:
# 1. Initialize the extractor (set to 50Hz)
gps_pipeline = GPSFeatureExtractor(hz=50, is_ms=True)

# 2. Point it at your folder of 6 months of data
master_gps_df = gps_pipeline.process_directory("data/objective-TeamB-2020/2020")
# single_gps_df = gps_pipeline.process_single_file(file_path='data/objective-TeamB-2020/2020/2020-06/2020-06-01/2020-06-01-TeamB-2f23d7d5-2326-49ce-b9c8-5a6303f785c5.parquet')

print(master_gps_df.head())
# single_gps_df

Found 1837 Parquet files across all months. Starting batch extraction...
  -> Processed 50 / 1837 files...
  -> Processed 100 / 1837 files...
  -> Processed 150 / 1837 files...
  -> Processed 200 / 1837 files...
  -> Processed 250 / 1837 files...
  -> Processed 300 / 1837 files...
  -> Processed 350 / 1837 files...
  -> Processed 400 / 1837 files...
  -> Processed 450 / 1837 files...
  -> Processed 500 / 1837 files...
  -> Processed 550 / 1837 files...
  -> Processed 600 / 1837 files...
  -> Processed 650 / 1837 files...
  -> Processed 700 / 1837 files...
  -> Processed 750 / 1837 files...
  -> Processed 800 / 1837 files...
  -> Processed 850 / 1837 files...
  -> Processed 900 / 1837 files...
  -> Processed 950 / 1837 files...
  -> Processed 1000 / 1837 files...
  -> Processed 1050 / 1837 files...
  -> Processed 1100 / 1837 files...
  -> Processed 1150 / 1837 files...
  -> Processed 1200 / 1837 files...
  -> Processed 1250 / 1837 files...
  -> Processed 1300 / 1837 files...
  -> Proces

In [5]:
# Save to the current folder as a Parquet file
master_gps_df.to_parquet("data/master_gps_features.parquet", index=False)

print("✅ Saved as Parquet successfully!")

✅ Saved as Parquet successfully!


### 2.1 EDA (Apply after all joins)

In [ ]:
# eda_pipeline = MetricsEDA(df=master_gps_df)
# multivariate_anomalies = eda_pipeline.run_all()

In [ ]:
# def clip_gps_outliers(df: pd.DataFrame, columns: list, upper_q: float = 0.99) -> pd.DataFrame:
#     """Clips extreme upper outliers to eliminate GPS satellite glitches."""
#     df_clipped = df.copy()
#     print(f"Clipping GPS artifacts at the {upper_q} quantile...")
    
#     for col in columns:
#         if col not in df_clipped.columns:
#             continue
#         upper_bound = df_clipped[col].quantile(upper_q)
#         clipped_high = (df_clipped[col] > upper_bound).sum()
        
#         if clipped_high > 0:
#             print(f"  -> {col}: Capped {clipped_high} extreme values.")
#         df_clipped[col] = df_clipped[col].clip(upper=upper_bound)
        
#     return df_clipped

# def engineer_gps_features_for_mlp(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Transforms and scales raw GPS features for a Neural Network backbone.
#     """
#     print("Pre-processing GPS data for MLP backbone...\n" + "-"*40)
#     df_dl = df.copy()
    
#     # 1. Feature Grouping
#     # These metrics are typically heavily right-skewed
#     skewed_gps_cols = [
#         'Distance', 'total_time_min', 'distance_per_min',
#         'sp_mir_t', 'sp_mir_d', 'sp_hir_t', 'sp_hir_d', 'sp_spr_t', 'sp_spr_d'
#     ]
#     # Proportions and bounded speeds
#     standard_gps_cols = [
#         'Speed_km_h_mean', 'Speed_km_h_max', 'Speed_km_h_std',
#         'sp_lir_p', 'sp_mir_p', 'sp_hir_p', 'sp_spr_p',
#         'sp_lir_t', 'sp_lir_d' # LIR usually forms the bulk of the session, less skewed
#     ]
    
#     # Filter columns to only those present in the dataframe
#     skewed_gps_cols = [col for col in skewed_gps_cols if col in df_dl.columns]
#     standard_gps_cols = [col for col in standard_gps_cols if col in df_dl.columns]
#     all_numeric_cols = skewed_gps_cols + standard_gps_cols

#     # 2. Clip GPS Artifacts BEFORE scaling
#     df_dl = clip_gps_outliers(df_dl, columns=all_numeric_cols, upper_q=0.99)
#     print("✅ Winsorized raw GPS artifacts.")

#     # 3. Transform Skewed Volume & High-Intensity Metrics
#     if skewed_gps_cols:
#         power_scaler = PowerTransformer(method='yeo-johnson', standardize=True)
#         df_dl[skewed_gps_cols] = power_scaler.fit_transform(df_dl[skewed_gps_cols])
#         print(f"✅ Applied Yeo-Johnson transform & scaling to {len(skewed_gps_cols)} skewed metrics.")

#     # 4. Standardize Remaining Features (Speeds & Proportions)
#     if standard_gps_cols:
#         standard_scaler = StandardScaler()
#         df_dl[standard_gps_cols] = standard_scaler.fit_transform(df_dl[standard_gps_cols])
#         print(f"✅ Applied Standard Scaling to {len(standard_gps_cols)} standard metrics.")
        
#     print("-"*40 + "\n✅ GPS Data is perfectly scaled for DeepHit tensors.")

#     return df_dl

# # Add has sprinted and has high intense sprint (has_hir) for model to understand large 0 spikes
# master_gps_df['has_sprinted'] = (master_gps_df['sp_spr_d'] > 0).astype(int)
# master_gps_df['has_hir'] = (master_gps_df['sp_hir_d'] > 0).astype(int)

# # Execute:
# nn_ready_gps_df = engineer_gps_features_for_mlp(master_gps_df)
# nn_ready_gps_df.head()

In [ ]:
# eda_pipeline = MetricsEDA(df=nn_ready_gps_df)
# multivariate_anomalies = eda_pipeline.run_all()